# Module 3.3 — Mesh Generation

**Why meshing is 40–60% of the engineer's time in a real CFD project:**
A perfect solver on a bad mesh gives wrong answers. A decent solver on a well-designed mesh gives good answers. In industrial CFD, mesh generation is often the bottleneck — not the solver. Understanding what makes a mesh good or bad is not optional.

**The map analogy:**
A mesh is a map of the fluid domain. Just as a good map must cover every part of the territory without gaps, overlaps, or distortion, a good mesh must cover every part of the fluid domain with non-overlapping cells of reasonable shape. The challenge: physical domains are curved, irregular, and multi-scale.

**Roadmap:**
1. Structured vs. unstructured meshes — advantages and trade-offs
2. Mesh quality metrics — skewness, aspect ratio, orthogonality, and why each matters
3. Near-wall meshing — $y^+$ estimation and boundary layer grading
4. Mesh convergence study — the only way to know if your mesh is fine enough
5. Common mesh topologies — O-grid, C-grid, H-grid
6. Python: generating structured meshes and measuring quality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay

## 1. Structured vs. Unstructured Meshes

### Structured mesh

Grid points arranged in a regular $i$-$j$-$k$ index space. Every interior point has exactly the same number of neighbours. Examples: Cartesian grids, cylindrical grids, body-fitted C-grids.

```
j=3  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
j=2  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─     Each cell: u[j,i] has neighbours
j=1  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─     u[j,i±1] and u[j±1,i] always
j=0  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
      i=0            i=nx
```

**Pros:** fast indexing (array slices, no lookup tables), cache-friendly, high-order schemes straightforward, fast solver setup.

**Cons:** hard to fit complex geometries; must maintain structured topology even when shape is irregular.

### Unstructured mesh

Cells of arbitrary shape (triangles, quads in 2D; tetrahedra, hexahedra in 3D) connected via a connectivity table. No regular index structure — each cell stores a list of its neighbours.

**Pros:** fits any geometry automatically; automatic refinement where needed (more cells in complex regions).

**Cons:** lower cache efficiency (random memory access), more complex data structures, harder to implement high-order schemes.

### Hybrid mesh — best of both

Structured boundary-layer cells (prism layers) near walls + unstructured tetrahedra in the bulk. This is the standard in industrial CFD:

```
Wall ══════════════════
      ║║║║║║║║║║║║║║║ ← Structured prism layers (y+ control, BL resolved)
      ║║║║║║║║║║║║║║║
    /\/\/\/\/\/\/\/\   ← Unstructured tetrahedra (fill the bulk)
```

| Property | Structured | Unstructured | Hybrid |
|----------|-----------|-------------|-------|
| Geometry fit | ❌ Hard | ✅ Easy | ✅ Easy |
| Near-wall accuracy | ✅ Good | ⚠️ Requires care | ✅ Best |
| Memory/speed | ✅ Fast | Slower | Moderate |
| Industry use | Simple geometry | Complex geometry | Most industrial cases |

## 2. Mesh Quality Metrics

### Skewness — shape deviation from ideal

How far does a cell deviate from the ideal shape (equilateral triangle or square)?

$$\text{Skewness} = \frac{\theta_{\max} - \theta_{\text{ideal}}}{180° - \theta_{\text{ideal}}}$$

where $\theta_{\text{ideal}} = 60°$ for triangles, $90°$ for quads.

- $0$: perfect equilateral cell
- $> 0.85$: problematic — interpolation error grows, solver may diverge
- $\to 1$: degenerate (flat, zero-volume cell — immediate divergence)

### Aspect ratio — elongation

$$AR = \frac{\text{longest edge}}{\text{shortest edge}}$$

High aspect ratio is **acceptable** in boundary layers (cells stretched in the wall-normal direction, very thin). It is **bad** in the bulk flow where isotropic diffusion occurs in all directions equally.

Practical guideline: $AR < 100$ in boundary layers, $AR < 5$ in bulk flow.

### Non-orthogonality — angle between face normal and cell-centre vector

$$\phi_{\text{non-orth}} = \angle(\hat{n}_{\text{face}}, \mathbf{d}_{\text{cells}})$$

For FVM, the diffusion flux is computed as $\Gamma(\phi_N - \phi_P)/|\mathbf{d}|$. This assumes $\hat{n}$ is parallel to $\mathbf{d}$. Non-orthogonality introduces a correction term:

$$F_{\text{total}} = F_{\text{orthogonal}} + F_{\text{non-orthogonal correction}}$$

Non-orthogonality $> 70°$: the correction dominates — solver diverges or requires very aggressive under-relaxation. OpenFOAM warns you at $> 70°$.

### Summary of quality targets

| Metric | Excellent | Acceptable | Problematic |
|--------|-----------|-----------|-------------|
| Skewness | $< 0.25$ | $< 0.85$ | $> 0.85$ |
| Aspect ratio (bulk) | $< 2$ | $< 5$ | $> 20$ |
| Non-orthogonality | $< 20°$ | $< 70°$ | $> 70°$ |
| $y^+$ (first cell, resolved BL) | $< 1$ | $1$–$5$ | $5$–$30$ |
| $y^+$ (first cell, wall function) | $30$–$60$ | $60$–$100$ | $> 200$ |

In [ ]:
# ── Mesh quality metrics: skewness of triangles ───────────────────────────────

def triangle_quality(p1, p2, p3):
    """Compute skewness and aspect ratio of a triangle."""
    # Side lengths
    a = np.linalg.norm(p2 - p3)
    b = np.linalg.norm(p1 - p3)
    c = np.linalg.norm(p1 - p2)
    # Angles via law of cosines
    cos_A = np.clip((b**2 + c**2 - a**2) / (2*b*c + 1e-12), -1, 1)
    cos_B = np.clip((a**2 + c**2 - b**2) / (2*a*c + 1e-12), -1, 1)
    cos_C = np.clip((a**2 + b**2 - c**2) / (2*a*b + 1e-12), -1, 1)
    angles = np.degrees([np.arccos(cos_A), np.arccos(cos_B), np.arccos(cos_C)])
    theta_max = angles.max()
    theta_min = angles.min()
    eq = 60.0  # equilateral = 60°
    skewness = max((theta_max - eq)/(180-eq), (eq - theta_min)/eq)
    ar = max(a, b, c) / min(a, b, c)
    return skewness, ar, angles

# Test cases
triangles = [
    ('Equilateral (ideal)',
     np.array([0.0,0.0]), np.array([1.0,0.0]), np.array([0.5,0.866])),
    ('Right angle (acceptable)',
     np.array([0.0,0.0]), np.array([1.0,0.0]), np.array([0.0,1.0])),
    ('Moderately skewed',
     np.array([0.0,0.0]), np.array([1.0,0.0]), np.array([0.3,0.5])),
    ('Highly skewed (near-degenerate)',
     np.array([0.0,0.0]), np.array([1.0,0.0]), np.array([0.95,0.05])),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, (name, p1, p2, p3) in zip(axes, triangles):
    sk, ar, angs = triangle_quality(p1, p2, p3)
    pts = np.array([p1, p2, p3, p1])
    color = '#2ecc71' if sk < 0.5 else '#f39c12' if sk < 0.8 else '#e74c3c'
    ax.fill(pts[:,0], pts[:,1], alpha=0.25, color=color)
    ax.plot(pts[:,0], pts[:,1], '-', color=color, lw=2)
    ax.set_title(f'{name}\nSkewness={sk:.3f} AR={ar:.1f}', fontsize=8)
    ax.set_aspect('equal'); ax.axis('off')
    # Colour legend
    label = '✓ Excellent' if sk < 0.5 else '⚠ Acceptable' if sk < 0.85 else '✗ Poor'
    ax.text(0.5, -0.12, label, ha='center', transform=ax.transAxes,
            fontsize=8, color=color, fontweight='bold')

plt.suptitle('Triangle skewness: shape quality metric (0=perfect, >0.85=problematic)',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── y+ calculator: first cell height for turbulent BL meshing ────────────────
# Uses flat-plate turbulent BL correlation: Cf = 0.027*Re^(-1/7)

def first_cell_height(Re_L, L, nu, y_plus_target, rho=1.0):
    """
    Estimate first cell height to achieve target y+ on a turbulent flat plate.
    Re_L: Reynolds number based on plate length
    L:    plate length [m]
    nu:   kinematic viscosity [m²/s]
    """
    U_inf = Re_L * nu / L
    Cf    = 0.027 * Re_L**(-1/7)      # skin friction coefficient
    tau_w = 0.5 * rho * U_inf**2 * Cf
    u_tau = np.sqrt(tau_w / rho)
    y1    = y_plus_target * nu / u_tau
    return y1, u_tau, Cf

# Mesh grading: exponential stretching ratio
def graded_mesh(y_first, H, n_layers, ratio):
    """Generate boundary-layer graded mesh: each cell is 'ratio' times thicker."""
    y = [0.0]
    dy = y_first
    for _ in range(n_layers):
        y.append(y[-1] + dy)
        dy *= ratio
    return np.array(y)

print('=== First cell height for y+ = 1 and y+ = 50 ===')
print(f'{"Case":<35} {"Re":>10} {"y+ target":>10} {"y1 (mm)":>12} {"u_tau":>10}')
print('-'*80)

cases = [
    ('Aircraft wing  (10m, air)',   1e7,  10.0, 1.5e-5, 1),
    ('Aircraft wing  (10m, air)',   1e7,  10.0, 1.5e-5, 50),
    ('Car (4m, air)',               5e6,   4.0, 1.5e-5, 1),
    ('Car (4m, air)',               5e6,   4.0, 1.5e-5, 50),
    ('Pipe (1m, water)',            1e5,   1.0, 1.0e-6, 1),
    ('Pipe (1m, water)',            1e5,   1.0, 1.0e-6, 30),
]

for name, Re, L, nu, yp_target in cases:
    y1, u_tau, Cf = first_cell_height(Re, L, nu, yp_target)
    print(f'{name:<35} {Re:>10.0e} {yp_target:>10.0f} {y1*1000:>12.4f} {u_tau:>10.4f}')

# Visualise a graded BL mesh
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# y+ distribution for a graded mesh
nu_pipe = 1e-6;  Re_pipe = 1e5;  L_pipe = 1.0
y1_pipe, u_tau_pipe, _ = first_cell_height(Re_pipe, L_pipe, nu_pipe, 1.0)
y_mesh = graded_mesh(y1_pipe, L_pipe/2, 40, 1.15)
y_plus_mesh = y_mesh * u_tau_pipe / nu_pipe

axes[0].plot(y_plus_mesh, np.arange(len(y_plus_mesh)), 'b-o', markersize=4)
axes[0].axvline(1,   color='green', ls='--', lw=1.5, label='y+=1 (sublayer limit)')
axes[0].axvline(5,   color='orange', ls='--', lw=1.5, label='y+=5 (buffer start)')
axes[0].axvline(30,  color='red',   ls='--', lw=1.5, label='y+=30 (log-law start)')
axes[0].set_xlabel('$y^+$'); axes[0].set_ylabel('Layer number')
axes[0].set_title(f'Graded BL mesh: first cell y1={y1_pipe*1e6:.1f} μm\nratio=1.15 per layer')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Visualise mesh cells
for y_edge in y_mesh[:15]:
    axes[1].axhline(y_edge, color='steelblue', lw=0.8)
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, y_mesh[14])
axes[1].set_title('Graded mesh cells (first 14 layers)\nCells thicken rapidly away from wall')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y (m)')
axes[1].text(0.5, 0, 'WALL', ha='center', va='bottom', fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Mesh Convergence Study — The Only Reliable Accuracy Check

### Why you must always do this

A single mesh gives you one answer. You don't know if that answer is converged to the true solution or still changing with grid refinement. Only by comparing three or more grids can you:
1. Confirm that the solution is converging (error decreasing with refinement)
2. Estimate the order of convergence (should match the scheme order, e.g. 2 for central differences)
3. Determine the mesh-independent solution

### Richardson extrapolation

If you have solutions on three grids (coarse, medium, fine) with grid spacing ratio $r$:

$$\phi_{\text{exact}} \approx \phi_f + \frac{\phi_f - \phi_m}{r^p - 1}$$

where $p$ is the observed order of convergence:

$$p = \frac{\ln\left[(\phi_m - \phi_c)/(\phi_f - \phi_m)\right]}{\ln r}$$

**Practical rule for industrial work:** if the quantity of interest changes by less than 1% between medium and fine grid, the medium grid is mesh-independent.

## 4. Common Mesh Topologies for External Flow

| Topology | Description | Best for |
|----------|-------------|----------|
| **O-grid** | Rings around the body | Circular/symmetric bodies (cylinder, sphere) |
| **C-grid** | Wraps around leading edge, cuts behind | Airfoils — captures BL on both surfaces |
| **H-grid** | Rectangular, body inside | Far-field regions, Cartesian IB method |
| **Hybrid C+H** | C-grid near wing + H far-field | Practical aircraft computations |

The **O-grid** (Module 1.7 cylinder example) is ideal for body-fitted cylinder meshes: the circular rings provide excellent orthogonality near the surface.

## Summary

| Concept | Key rule |
|---------|----------|
| Skewness | $< 0.85$; avoid buffer-layer $y^+$ (5–30) |
| Aspect ratio | High in BL (ok), low in bulk (required) |
| Non-orthogonality | $< 70°$; OpenFOAM warns above this |
| $y^+$ | $< 1$ for resolved BL; $30$–$100$ for wall functions |
| Grading ratio | $< 1.2$ per layer in BL — no sudden size jumps |
| Convergence study | Always: coarse / medium / fine — report Richardson extrapolation |

---
**Next:** Module 3.4 — Higher-Order Schemes and TVD Limiters: second-order accuracy without oscillations near shocks.